<div style="background: linear-gradient(135deg, #0f2027, #203a43, #2c5364); padding: 36px 32px; border-radius: 14px; margin-bottom: 4px;">
  <h1 style="color: #ffffff; font-size: 2.2em; margin: 0 0 8px 0; font-weight: 800;">🧠 Hackathon Unboxed 2026</h1>
  <p style="color: #94b4c1; font-size: 1.05em; margin: 0 0 22px 0;">Plateforme IA Médicale &nbsp;·&nbsp; GPU Node 1 &nbsp;·&nbsp; 28 février 2026</p>
  <div style="display:flex; gap:10px; flex-wrap:wrap;">
    <span style="background:#e94560; color:#fff; padding:4px 16px; border-radius:20px; font-size:.82em; font-weight:700;">PyTorch 2.10</span>
    <span style="background:rgba(255,255,255,.08); color:#64ffda; border:1px solid #64ffda; padding:4px 16px; border-radius:20px; font-size:.82em; font-weight:700;">MONAI · pydicom · SimpleITK</span>
    <span style="background:rgba(255,255,255,.08); color:#64ffda; border:1px solid #64ffda; padding:4px 16px; border-radius:20px; font-size:.82em; font-weight:700;">Orthanc DICOM</span>
  </div>
</div>

> 📌 **Ce notebook est votre point de départ.** Exécutez les cellules dans l'ordre pour valider votre environnement avant de commencer à coder.

---

## 📋 Vos ressources

| Ressource | Valeur |
|-----------|--------|
| 🌐 **JupyterLab** | `https://jupyter.unboxed-2026.ovh/<votre-équipe>` |
| 🏥 **Orthanc (DICOM)** | `https://orthanc.unboxed-2026.ovh` |
| 🔑 **Orthanc credentials** | `unboxed` / `unboxed2026` |
| 🖥️ **CPU** | 4 cœurs dédiés |
| 💾 **RAM** | 12 Go |
| ⚡ **GPU** | NVIDIA (CUDA 12.3) |
| 📂 **Dataset** | `/home/jovyan/dataset` *(lecture seule)* |
| 💼 **Votre espace** | `/home/jovyan/work` *(persistant)* |

---

## 🗺️ Contenu de ce notebook

| Étape | Section | Objectif |
|-------|---------|----------|
| ① | Python & PyTorch | Vérifier l'environnement de base |
| ② | Librairies médicales | pydicom, MONAI, SimpleITK, nibabel… |
| ③ | Dataset | Explorer les données disponibles |
| ④ | Connexion Orthanc | Tester l'accès au serveur DICOM |
| ⑤ | Lister les examens | Voir les études disponibles |
| ⑥ | Upload DICOM | Envoyer des fichiers vers Orthanc |
| ⑦ | Download | Télécharger une étude complète |
| ⑧ | Visualisation | Afficher une coupe DICOM |
| ⑨ | Benchmark | Tester la puissance de calcul |

---
## ① &nbsp; Python & PyTorch
Vérifie la version Python, PyTorch et la disponibilité du GPU.

In [4]:
pip install torch 

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_curand_cu12-10.3.9.90-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cusolver_c

In [2]:
import torch, sys


print("─" * 45)
print(f"  Python      : {sys.version.split()[0]}")
print(f"  PyTorch     : {torch.__version__}")
print(f"  CUDA dispo  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU         : {props.name}")
    print(f"  VRAM        : {props.total_memory / 1e9:.1f} Go")
else:
    print("  GPU         : non disponible (CPU mode)")
print("─" * 45)

ModuleNotFoundError: No module named 'torch'

---
## ② &nbsp; Librairies médicales
Vérifie que toutes les librairies d'imagerie médicale sont bien installées.

In [3]:
import importlib

libs = [
    ("pydicom",    "pydicom"),
    ("SimpleITK",  "SimpleITK"),
    ("nibabel",    "nibabel"),
    ("monai",      "monai"),
    ("cv2",        "opencv-python"),
    ("skimage",    "scikit-image"),
    ("dicom2nifti","dicom2nifti"),
    ("pynetdicom", "pynetdicom"),
    ("pandas",     "pandas"),
    ("matplotlib", "matplotlib"),
    ("plotly",     "plotly"),
]

print("─" * 40)
all_ok = True
for mod, label in libs:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "✓")
        print(f"  {'✅':<3} {label:<20} {ver}")
    except ImportError:
        print(f"  {'❌':<3} {label:<20} NON INSTALLÉ")
        all_ok = False
print("─" * 40)
print("  Toutes les librairies sont OK ✓" if all_ok else "  ⚠️  Des librairies manquent — contacter les organisateurs")

────────────────────────────────────────
  ❌   pydicom              NON INSTALLÉ
  ❌   SimpleITK            NON INSTALLÉ
  ❌   nibabel              NON INSTALLÉ
  ❌   monai                NON INSTALLÉ
  ❌   opencv-python        NON INSTALLÉ
  ❌   scikit-image         NON INSTALLÉ
  ❌   dicom2nifti          NON INSTALLÉ
  ❌   pynetdicom           NON INSTALLÉ
  ❌   pandas               NON INSTALLÉ
  ❌   matplotlib           NON INSTALLÉ
  ❌   plotly               NON INSTALLÉ
────────────────────────────────────────
  ⚠️  Des librairies manquent — contacter les organisateurs


---
## ③ &nbsp; Dataset
Explore le contenu du dossier `/home/jovyan/dataset` (lecture seule).

In [7]:
from pathlib import Path

dataset = Path("/home/jovyan/dataset")
print("─" * 50)
if not dataset.exists():
    print("  ⚠️  Dataset non monté — contacter les organisateurs")
else:
    items = sorted(dataset.iterdir())
    dirs  = [p for p in items if p.is_dir()]
    files = [p for p in items if p.is_file()]
    print(f"  📂 Dataset     : {dataset}")
    print(f"  📁 Dossiers    : {len(dirs)}")
    print(f"  📄 Fichiers    : {len(files)}")
    print(f"  Total          : {len(items)} éléments")
    print()
    for p in sorted(items)[:15]:
        icon = "📁" if p.is_dir() else "📄"
        print(f"    {icon}  {p.name}")
    if len(items) > 15:
        print(f"    … et {len(items)-15} autre(s)")
print("─" * 50)

──────────────────────────────────────────────────
  📂 Dataset     : /home/jovyan/dataset
  📁 Dossiers    : 0
  📄 Fichiers    : 0
  Total          : 0 éléments

──────────────────────────────────────────────────


---
## ④ &nbsp; Connexion à Orthanc

| Champ | Valeur |
|-------|--------|
| URL interne | `http://10.0.1.215:8042` |
| URL publique | `https://orthanc.unboxed-2026.ovh` |
| Utilisateur | `unboxed` |
| Mot de passe | `unboxed2026` |
| AET DICOM | `UNBOXED` · Port `4242` |

In [ ]:
import requests

ORTHANC = "http://10.0.1.215:8042"
AUTH    = ("unboxed", "unboxed2026")

print("─" * 50)
try:
    r = requests.get(f"{ORTHANC}/system", auth=AUTH, timeout=5)
    if r.status_code == 200:
        info = r.json()
        print(f"  ✅ Orthanc connecté")
        print(f"  Version     : {info['Version']}")
        print(f"  AET         : {info['DicomAet']}")
        print(f"  Storage     : {info.get('StorageSize', 'N/A')}")
    else:
        print(f"  ❌ Erreur HTTP {r.status_code}")
except requests.exceptions.ConnectionError:
    print("  ❌ Impossible de joindre Orthanc (réseau privé requis)")
except requests.exceptions.Timeout:
    print("  ❌ Timeout — serveur Orthanc non joignable")
print("─" * 50)

---
## ⑤ &nbsp; Lister les examens DICOM disponibles

In [13]:
import requests, pandas as pd

ORTHANC = "http://10.0.1.215:8042"
AUTH    = ("unboxed", "unboxed2026")

studies_ids = requests.get(f"{ORTHANC}/studies", auth=AUTH, timeout=5).json()
print(f"  📊 {len(studies_ids)} étude(s) DICOM dans Orthanc\n")

if not studies_ids:
    print("  ℹ️  Le dataset sera chargé avant le hackathon.")
else:
    rows = []
    for sid in studies_ids[:10]:
        info = requests.get(f"{ORTHANC}/studies/{sid}", auth=AUTH).json()
        t = info.get("MainDicomTags", {})
        rows.append({
            "ID"          : sid[:12] + "…",
            "Patient"     : t.get("PatientID", "-"),
            "Description" : t.get("StudyDescription", "-"),
            "Date"        : t.get("StudyDate", "-"),
            "Modalité"    : t.get("ModalitiesInStudy", "-"),
        })
    df = pd.DataFrame(rows)
    display(df)

  📊 2 étude(s) DICOM dans Orthanc



,ID,Patient,Description,Date,Modalité
0,b40811fa-96f…,-,TC TÒRAX,20210405,-
1,c2f78efd-5cc…,-,TC TÒRAX,20200503,-


---
## ⑥ &nbsp; Upload DICOM vers Orthanc

Deux fonctions utilitaires : upload d'un fichier unique ou d'un dossier entier.

In [14]:
import requests
from pathlib import Path

ORTHANC = "http://10.0.1.215:8042"
AUTH    = ("unboxed", "unboxed2026")

def upload_dicom(path: str) -> str | None:
    """Envoie un fichier .dcm vers Orthanc. Retourne l'instance ID."""
    with open(path, "rb") as f:
        r = requests.post(f"{ORTHANC}/instances", auth=AUTH,
                          data=f.read(), headers={"Content-Type": "application/dicom"})
    if r.status_code == 200:
        iid = r.json().get("ID")
        print(f"  ✅ Upload OK  →  instance ID : {iid}")
        return iid
    print(f"  ❌ Erreur {r.status_code} : {r.text}")
    return None

def upload_dicom_folder(folder: str) -> None:
    """Upload tous les .dcm d'un dossier (récursif) vers Orthanc."""
    files = list(Path(folder).rglob("*.dcm"))
    print(f"  📂 {len(files)} fichier(s) .dcm trouvé(s) dans '{folder}'")
    print("─" * 50)
    ok = err = 0
    for f in files:
        with open(f, "rb") as fh:
            r = requests.post(f"{ORTHANC}/instances", auth=AUTH,
                              data=fh.read(), headers={"Content-Type": "application/dicom"})
        if r.status_code == 200:
            print(f"  ✅  {f.name}")
            ok += 1
        else:
            print(f"  ❌  {f.name}  (HTTP {r.status_code})")
            err += 1
    print("─" * 50)
    print(f"  Résultat : {ok} OK · {err} erreur(s)")

# ── Utilisation ──────────────────────────────────────────
# upload_dicom("/home/jovyan/dataset/mon_examen.dcm")
# upload_dicom_folder("/home/jovyan/dataset/CT_series")
print("  Fonctions upload_dicom() et upload_dicom_folder() prêtes ✓")

  Fonctions upload_dicom() et upload_dicom_folder() prêtes ✓


---
## ⑦ &nbsp; Télécharger une étude depuis Orthanc

Obtenir l'`study_id` depuis la cellule ⑤, puis appeler `download_study()`.

In [15]:
import requests
from pathlib import Path

ORTHANC = "http://10.0.1.215:8042"
AUTH    = ("unboxed", "unboxed2026")

def download_study(study_id: str, out_dir: str = "/home/jovyan/work") -> str:
    """Télécharge une étude complète (.zip) depuis Orthanc."""
    dest = Path(out_dir) / f"study_{study_id[:8]}.zip"
    print(f"  ⬇️  Téléchargement de l'étude {study_id[:12]}…")
    with requests.get(f"{ORTHANC}/studies/{study_id}/archive",
                      auth=AUTH, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = 0
        with open(dest, "wb") as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)
                total += len(chunk)
    print(f"  ✅ Sauvegardé : {dest}  ({total/1e6:.1f} Mo)")
    return str(dest)

# ── Utilisation ──────────────────────────────────────────────────────
# study_id = "1f8a3b2c-..."   # ← obtenu depuis la cellule ⑤
# download_study(study_id)
print("  Fonction download_study() prête ✓")

  Fonction download_study() prête ✓


---
## ⑧ &nbsp; Visualiser une image DICOM

Lit un fichier `.dcm` et affiche la coupe avec ses métadonnées.

In [16]:
import pydicom, numpy as np, matplotlib.pyplot as plt

def show_dicom(path: str):
    """Affiche une coupe DICOM avec ses métadonnées principales."""
    ds  = pydicom.dcmread(path)
    img = ds.pixel_array.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    fig, ax = plt.subplots(1, 1, figsize=(6, 6), facecolor="#111")
    ax.imshow(img, cmap="gray", interpolation="bilinear")
    ax.set_title(
        f"Patient: {getattr(ds,'PatientID','?')}  |  "
        f"Modality: {getattr(ds,'Modality','?')}  |  "
        f"Slice: {getattr(ds,'InstanceNumber','?')}",
        color="white", fontsize=10, pad=10
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f"  Dimensions   : {img.shape}")
    print(f"  Pixel spacing: {getattr(ds,'PixelSpacing','N/A')}")
    print(f"  Study date   : {getattr(ds,'StudyDate','N/A')}")
    return ds

# ── Utilisation ───────────────────────────────────────
# ds = show_dicom("/home/jovyan/dataset/exemple.dcm")
print("  Fonction show_dicom() prête ✓")

  Fonction show_dicom() prête ✓


---
## ⑨ &nbsp; Benchmark de calcul

Produit matriciel 4096×4096 — mesure la puissance disponible (GPU ou CPU).

In [17]:
import torch, time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device      : {device}")

A = torch.randn(4096, 4096, device=device)
B = torch.randn(4096, 4096, device=device)

# Warmup
_ = torch.mm(A, B)
if torch.cuda.is_available():
    torch.cuda.synchronize()

t0 = time.perf_counter()
C  = torch.mm(A, B)
if torch.cuda.is_available():
    torch.cuda.synchronize()
elapsed = (time.perf_counter() - t0) * 1000

print(f"  Matmul 4096×4096 sur {device} : {elapsed:.1f} ms")
if torch.cuda.is_available():
    print(f"  VRAM utilisée   : {torch.cuda.memory_allocated()/1e6:.0f} Mo")
print("  Benchmark OK ✓")

  Device      : cpu
  Matmul 4096×4096 sur cpu : 986.3 ms
  Benchmark OK ✓


---

## 📦 Librairies disponibles

| Librairie | Usage |
|-----------|-------|
| `torch` / `torchvision` | Deep Learning |
| `monai` | Segmentation médicale 3D |
| `pydicom` | Lecture / écriture DICOM |
| `SimpleITK` | Traitement d'images médicales |
| `nibabel` | Format NIfTI (IRM) |
| `dicom2nifti` | Conversion DICOM → NIfTI |
| `pynetdicom` | Protocole DICOM réseau (C-STORE, C-FIND…) |
| `opencv-python` | Vision par ordinateur |
| `scikit-image` | Traitement d'images |
| `pandas` / `matplotlib` / `seaborn` / `plotly` | Analyse & visualisation |
| `ipywidgets` | Widgets interactifs |

---

## ⚠️ Règles importantes

- 📂 `/home/jovyan/dataset` est **en lecture seule** — ne pas tenter d'y écrire.
- 💾 Sauvegardez uniquement dans `/home/jovyan/work` — c'est votre espace persistant.
- 🔒 Ne partagez pas vos credentials JupyterLab avec d'autres équipes.
- 🆘 Problème technique → Slack **#support-technique**

---

<div style="text-align:center; color:#555; font-size:0.85em; padding: 20px 0 8px 0;">
Hackathon Unboxed 2026 &nbsp;·&nbsp; GPU Node 1 &nbsp;·&nbsp; Bonne chance à toutes les équipes 🚀
</div>